<a href="https://colab.research.google.com/github/MunchProductionz/jepa-financial-time-series-ablation/blob/master/notebooks/colab_experiment_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JEPA Financial Time-Series Ablation Runner (Google Colab)

This notebook runs model-config ablations, saves all artifacts to Google Drive, loads the saved outputs for analysis, and exports LaTeX tables with metrics as columns. Cross-model comparison tables bold the best value per metric and append `($\uparrow$)` or `($\downarrow$)` to directional metric headers.

Use it in two modes:

- **Train + analyze**: clone/install the repo, prepare data, run configs, then analyze results.
- **Analyze only**: skip training and point `PREDICTIONS_DIR` at a previous Drive-backed `predictions` directory.

For LaTeX output, include `\usepackage{booktabs}` in your paper preamble.

## 1. Runtime and Paths

Set `REPO_URL` to your GitHub repo. For a private repo, either use a Colab secret/token in the URL or upload a zip/folder to Drive and set `REPO_URL = None` plus `REPO_DIR` to that uploaded path.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

USE_DRIVE = True
REPO_URL = "https://github.com/MunchProductionz/jepa-financial-time-series-ablation.git"  # set to None if repo is already available
BRANCH = "master"
REPO_DIR = Path("/content/jepa-financial-time-series-ablation")
DRIVE_ROOT = Path("/content/drive/MyDrive/jepa_financial_ablation")

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        print("google.colab is unavailable; continuing outside Colab.")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = DRIVE_ROOT / "data"
PREDICTIONS_DIR = DRIVE_ROOT / "predictions"
LATEX_DIR = DRIVE_ROOT / "latex_tables"
for path in (DATA_ROOT, PREDICTIONS_DIR, LATEX_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Predictions dir:", PREDICTIONS_DIR)

Mounted at /content/drive
Drive root: /content/drive/MyDrive/jepa_financial_ablation
Predictions dir: /content/drive/MyDrive/jepa_financial_ablation/predictions


## 2. Get Code Into Colab and Install

This installs the package editable so the Python API and CLI are available. If you uploaded the repo to Drive, set `REPO_URL = None` and change `REPO_DIR` to that folder before running this cell.

In [2]:
if REPO_URL and not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)

if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repo directory does not exist: {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Installed repo from", REPO_DIR)

Installed repo from /content/jepa-financial-time-series-ablation


## 3. Prepare Data

Choose one data mode:

- `smoke`: build deterministic synthetic panels inside Drive. Best for verifying the pipeline.
- `drive_existing`: use data you have already copied to Drive.
- `download_sp500`: run the repo's download commands in Colab. This can take a long time and depends on network/API availability.

In [ ]:
DATA_MODE = "smoke"  # values: smoke, drive_existing, download_sp500

if DATA_MODE == "smoke":
    smoke_short = DATA_ROOT / "prices" / "smoke" / "short" / "panel.csv"
    smoke_long = DATA_ROOT / "prices" / "smoke" / "long" / "panel.csv"
    smoke_short.parent.mkdir(parents=True, exist_ok=True)
    smoke_long.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "ablation-study-jepa", "build-sample-data",
        "--output", str(smoke_short),
        "--periods", "240",
        "--tickers", "AAPL,MSFT,NVDA,AMZN",
    ], check=True)
    subprocess.run([
        "ablation-study-jepa", "build-sample-data",
        "--output", str(smoke_long),
        "--periods", "720",
        "--tickers", "AAPL,MSFT,NVDA,AMZN,GOOGL,META,JPM,XOM",
    ], check=True)
    DATA_OVERRIDES = {
        "data.data_dir": str(smoke_short),
        "data.macro_data_path": None,
        "data.macro_feature_columns": [],
        "data.limit": None,
        "data.start_date": None,
        "data.end_date": None,
        "data.fast_feature_columns": [
            "Price Open", "Price High", "Price Low", "Price Close", "Price Adj_Close", "Volume",
        ],
        "data.slow_feature_columns": [
            "beta", "pe_ratio", "debt_to_equity", "interest_rate", "vix",
        ],
        "features.sequence": [
            "Price Open", "Price High", "Price Low", "Price Close", "Price Adj_Close", "Volume",
            "beta", "pe_ratio", "debt_to_equity", "interest_rate", "vix",
            "return_1d", "return_5d", "return_20d", "volatility_20d", "volume_zscore",
        ],
        "features.static": [
            "sector_consumer", "sector_semiconductors", "sector_technology", "sector_unknown",
        ],
        "splits.method": "fraction",
        "splits.train": 0.6,
        "splits.validation": 0.2,
        "splits.test": 0.2,
        "sliding_window.enabled": False,
        "dataset.lookback": 24,
        "dataset.batch_size": 16,
        "dataset.num_workers": 0,
        "dataset.persistent_workers": False,
        "model.hidden_dim": 16,
        "model.num_transformer_blocks": 2,
        "model.num_attention_heads": 4,
        "model.dropout": 0.0,
        "jepa.projection_dim": 16,
        "jepa.horizons": [1, 5],
        "training.max_epochs": 1,
        "training.early_stopping": False,
    }
elif DATA_MODE == "drive_existing":
    DATA_OVERRIDES = {
        "data.data_dir": str(DATA_ROOT / "prices" / "sp500" / "data"),
        "data.macro_data_path": str(DATA_ROOT / "macro" / "fred_md" / "fred_md_1960_2025.csv"),
    }
elif DATA_MODE == "download_sp500":
    raise NotImplementedError(
        "Run the README download commands here, then set DATA_MODE='drive_existing'. "
        "Large downloads are intentionally not started automatically."
    )
else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE}")

DATA_OVERRIDES["evaluation.predictions_dir"] = str(PREDICTIONS_DIR)
DATA_OVERRIDES

## 4. Choose Configs and Run Experiments

Use smoke configs first. For real runs, switch to the full configs after the smoke matrix succeeds.

In [ ]:
CONFIGS = [
    "configs/exp/tft.yaml",
    "configs/exp/contrastive_jepa_ablation.yaml",
    "configs/exp/lejepa_fixed_base.yaml",
    "configs/exp/lejepa_fixed_predictive_only.yaml",
    "configs/exp/lejepa_fixed_sigreg_only_direct_h.yaml",
    "configs/exp/lejepa_fixed_sigreg_only_adapter_whitened.yaml",
]

METRIC_COLUMNS = [
    "rmse",
    "mae",
    "spearman_rank_ic",
    "directional_accuracy",
    "long_short_decile_return",
    "long_short_decile_sharpe",
    "long_short_decile_turnover",
    "long_short_decile_transaction_cost_adjusted_return",
]

EVALUATION_OVERRIDES = {
    "evaluation.metrics": METRIC_COLUMNS,
    "evaluation.portfolio_quantile": 0.1,
    "evaluation.transaction_cost_bps": 10.0,
}

TRAINING_OVERRIDES = {
    "training.accelerator": "auto",  # use "gpu" when Colab GPU is available
    "training.devices": "auto",
    "logging.wandb.enabled": False,
}

RUN_EXPERIMENTS = True  # set False for analysis-only mode

from ablation_study_jepa.api.experiment import ExperimentRunner
from ablation_study_jepa.config.loader import load_config

results = []
if RUN_EXPERIMENTS:
    for config_path in CONFIGS:
        overrides = {**DATA_OVERRIDES, **EVALUATION_OVERRIDES, **TRAINING_OVERRIDES}
        config = load_config(config_path, overrides=overrides)
        print(f"Running {config_path} -> {config.run_name}")
        result = ExperimentRunner(config).run()
        results.append(result)
        print("metrics:", result.metrics_path)
else:
    print("Skipping training. Analysis will use existing artifacts in", PREDICTIONS_DIR)

## 5. Load Results and Build Comparison Tables

The following cells work both immediately after training and later in an analysis-only session.

In [ ]:
from ablation_study_jepa.evaluation.compare_runs import (
    comparison_metrics_frame,
    completed_runs,
    load_analysis_tables,
    discover_run_dirs,
)

METRIC_COLUMNS = [
    "rmse",
    "mae",
    "spearman_rank_ic",
    "directional_accuracy",
    "long_short_decile_return",
    "long_short_decile_sharpe",
    "long_short_decile_turnover",
    "long_short_decile_transaction_cost_adjusted_return",
]

manifest = completed_runs(PREDICTIONS_DIR)
display(manifest.tail() if not manifest.empty else manifest)

test_comparison = comparison_metrics_frame(
    PREDICTIONS_DIR,
    split="test",
    metric_names=METRIC_COLUMNS,
)
val_comparison = comparison_metrics_frame(
    PREDICTIONS_DIR,
    split="val",
    metric_names=METRIC_COLUMNS,
)

display(test_comparison)
display(val_comparison)

## 6. Export LaTeX Tables

The generated comparison tables use metrics as columns, bold the best directional value in each metric column, and add `($\uparrow$)` or `($\downarrow$)` to metric headers.

In [ ]:
from ablation_study_jepa.evaluation.latex_tables import (
    comparison_latex_table,
    export_study_latex_tables,
)

latex_paths = export_study_latex_tables(
    predictions_dir=PREDICTIONS_DIR,
    output_dir=LATEX_DIR,
    metric_columns=METRIC_COLUMNS,
    splits=("val", "test"),
)
latex_paths

In [ ]:
test_latex = comparison_latex_table(
    test_comparison,
    metric_columns=METRIC_COLUMNS,
    caption="Test metric comparison across model configs.",
    label="tab:test_metric_comparison",
)
print(test_latex)

print("LaTeX files written to:", LATEX_DIR)

## 7. Optional Deeper Analysis

Load per-run analysis artifacts for custom plots. These files are enough to rerun analysis later without retraining.

In [ ]:
run_dirs = discover_run_dirs(PREDICTIONS_DIR)
print("runs:", len(run_dirs))
if run_dirs:
    tables = load_analysis_tables(run_dirs[-1])
    print("tables:", sorted(tables))
    if "portfolio_returns" in tables:
        display(tables["portfolio_returns"].head())
    if "per_date_metrics" in tables:
        display(tables["per_date_metrics"].head())